# Machine Learning 1 — Homework 1, Question 5

This notebook completes **Q5 only**, with the required classification and regression comparisons. Four simple models use PyTorch automatic differentiation and mini-batch SGD; the two scikit-learn baselines use their native fitting methods. There are no hidden layers or extra datasets.

- **Classification:** scikit-learn's breast cancer dataset, all 30 features, original 0/1 target encoding. The 80/20 split uses `random_state=42` and `stratify=y`. All three classifiers share the same training and test indices. Accuracy is the test metric, with probability threshold 0.5 for both self-implemented models.
- **Regression:** the real [UCI ALE dataset 844](https://archive.ics.uci.edu/dataset/844/average+localization+error+(ale)+in+sensor+node+localization+process+in+wsns), CSV `mcs_ds_edited_iter_shuffled.csv`. Predict `ale` using exactly `anchor_ratio`, `trans_range`, `node_density`, and `iterations`; ignore `sd_ale` and any CSV index column. The 80/20 split uses `random_state=42`, without stratification, and directly reuses Q4's saved split. All three regressors share these indices. Test MAE is reported in the original `ale` units (meters, according to the UCI variable table).
- A separate `StandardScaler` is fitted **only on each dataset's training features**, then applied to its training and test features. All three models for that dataset share the scaled arrays. `ale` is neither scaled nor transformed.

The homework specifies the 80/20 splits. Input standardization, seed, mini-batch size, learning rate, number of epochs, clipping epsilon, and other training settings below are implementation choices; they are not claimed to be optimal. All settings are fixed before test evaluation. We retain the default regularization of the scikit-learn logistic baseline; the explicit BCE objectives contain no regularization term, so exact agreement is not expected.

**Reproduction:** keep this notebook with its `data/` directory, then run every cell from top to bottom. It requires NumPy, pandas, scikit-learn, PyTorch, and IPython. The included CSV is a copy of the official real data used for Q4; no synthetic replacement is permitted. The saved ALE split includes the CSV SHA-256 checksum. Missing or inconsistent data cause an explicit error. Numeric results and per-epoch training losses are saved in `results/`. No Q4 normality analyses are repeated.

In [1]:
from pathlib import Path
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import sklearn
import torch
from torch import nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
from IPython.display import display, Markdown

# All experimental training settings are centralized here.
CONFIG = {
    "seed": 42,
    "dtype": torch.float64,
    "device": "cpu",
    "learning_rate": 0.01,
    "batch_size": 32,
    "epochs": 1000,
    "momentum": 0,
    "weight_decay": 0,
    "probability_clip_eps": 1e-12,
    "cpu_threads": 1,
}
SPLIT_CONFIG = {"test_size": 0.2, "random_state": 42}
feature_columns = ["anchor_ratio", "trans_range", "node_density", "iterations"]
target_column = "ale"

# Execute the notebook with the kernel working directory set to this folder.
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.set_num_threads(CONFIG["cpu_threads"])

versions = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "PyTorch": torch.__version__,
}
print("Library versions:", versions)
print("Final training configuration:", CONFIG)
print("Split configuration:", SPLIT_CONFIG)

# JSON stores the dtype as text without changing the tensor dtype used below.
(RESULTS_DIR / "configuration.json").write_text(
    json.dumps({"training": {**CONFIG, "dtype": str(CONFIG["dtype"])},
                "split": SPLIT_CONFIG, "versions": versions}, indent=2)
)

Library versions: {'Python': '3.13.5', 'NumPy': '2.3.3', 'pandas': '2.3.3', 'scikit-learn': '1.8.0', 'PyTorch': '2.10.0'}
Final training configuration: {'seed': 42, 'dtype': torch.float64, 'device': 'cpu', 'learning_rate': 0.01, 'batch_size': 32, 'epochs': 1000, 'momentum': 0, 'weight_decay': 0, 'probability_clip_eps': 1e-12, 'cpu_threads': 1}
Split configuration: {'test_size': 0.2, 'random_state': 42}


463

## Data validation, shared splits, and preprocessing

The original classification labels are retained exactly as provided by `load_breast_cancer`; the printed `target_names` defines their meaning. UCI reports 107 ALE records. We print the actual shapes, columns, dtypes, and missing-value counts before fitting. The saved Q4 split is checked against its checksum and the specified `train_test_split` result, then reused in the same sample order.

Source credit: Singh, A., Kotiyal, V., Sharma, S., Nagar, J., & Lee, C. (2020). *Average Localization Error (ALE) in sensor node localization process in WSNs* [Dataset]. UCI Machine Learning Repository. [DOI: 10.24432/C5ZP6R](https://doi.org/10.24432/C5ZP6R), CC BY 4.0. The included official UCI CSV was downloaded for Q4 on October 14, 2026, then copied into this standalone Q5 bundle.

In [2]:
# Classification: all 30 inputs and original target labels.
cancer = load_breast_cancer()
X_class_all = np.asarray(cancer.data, dtype=np.float64)
y_class_all = np.asarray(cancer.target, dtype=np.int64).reshape(-1)
print("Classification target_names:", cancer.target_names)
print("Original label meanings:", dict(enumerate(cancer.target_names)))
print("Classification X shape:", X_class_all.shape, "y shape:", y_class_all.shape)
print("Classification feature names:", list(cancer.feature_names))
print("Classification dtypes:", X_class_all.dtype, y_class_all.dtype)
print("Classification missing values:", int(np.isnan(X_class_all).sum()),
      int(pd.isna(y_class_all).sum()))
assert X_class_all.shape == (569, 30)
assert np.array_equal(np.unique(y_class_all), np.array([0, 1]))
assert np.isfinite(X_class_all).all()

class_train_indices, class_test_indices = train_test_split(
    np.arange(y_class_all.size), stratify=y_class_all, **SPLIT_CONFIG
)
X_class_train_raw = X_class_all[class_train_indices]
X_class_test_raw = X_class_all[class_test_indices]
y_class_train = y_class_all[class_train_indices]
y_class_test = y_class_all[class_test_indices]

# Regression: use the exact real CSV and saved ALE split from Q4.
csv_path = DATA_DIR / "mcs_ds_edited_iter_shuffled.csv"
split_path = DATA_DIR / "ale_train_test_split.npz"
if not csv_path.is_file():
    raise FileNotFoundError(
        "The official ALE CSV is missing. Restore the real UCI dataset 844 CSV; "
        "this notebook does not substitute synthetic data."
    )
if not split_path.is_file():
    raise FileNotFoundError("The Q4 ALE split file is missing; restore data/ale_train_test_split.npz.")
ale_df = pd.read_csv(csv_path)
print("\nALE actual shape:", ale_df.shape, "; UCI reported samples: 107")
print("ALE column names:", list(ale_df.columns))
print("ALE dtypes:")
print(ale_df.dtypes.to_string())
print("ALE missing-value counts:")
print(ale_df.isna().sum().to_string())
expected_columns = feature_columns + [target_column, "sd_ale"]
if ale_df.shape != (107, 6) or list(ale_df.columns) != expected_columns:
    raise ValueError(
        "Actual ALE data differ from the validated official 107-by-6 CSV. "
        "Inspect the printed shape and columns; no automatic data modification is performed."
    )
if not all(pd.api.types.is_numeric_dtype(ale_df[c]) for c in expected_columns):
    raise TypeError("ALE columns must have numeric types; inspect the printed dtypes.")
if ale_df.isna().any().any():
    raise ValueError("ALE contains missing values; no automatic imputation or row deletion is performed.")
if not np.isfinite(ale_df.to_numpy(dtype=np.float64)).all():
    raise ValueError("ALE contains non-finite numeric values.")

X_reg_all = ale_df.loc[:, feature_columns].to_numpy(dtype=np.float64)
y_reg_all = ale_df.loc[:, target_column].to_numpy(dtype=np.float64).reshape(-1)
print("Regression model input shape:", X_reg_all.shape, "target shape:", y_reg_all.shape)
print("Regression inputs:", feature_columns, "; target:", target_column)

actual_sha256 = hashlib.sha256(csv_path.read_bytes()).hexdigest()
with np.load(split_path, allow_pickle=False) as saved_split:
    reg_train_indices = saved_split["train_indices"].astype(np.int64)
    reg_test_indices = saved_split["test_indices"].astype(np.int64)
    saved_test_size = float(saved_split["test_size"].item())
    saved_random_state = int(saved_split["random_state"].item())
    saved_sha256 = str(saved_split["csv_sha256"].item())
if actual_sha256 != saved_sha256:
    raise ValueError("ALE CSV checksum does not match the dataset used for the Q4 saved split.")
if saved_test_size != SPLIT_CONFIG["test_size"] or saved_random_state != SPLIT_CONFIG["random_state"]:
    raise ValueError("The saved ALE split settings do not match this experiment.")
expected_train_indices, expected_test_indices = train_test_split(
    np.arange(y_reg_all.size), **SPLIT_CONFIG  # Regression: no stratify.
)
assert np.array_equal(reg_train_indices, expected_train_indices)
assert np.array_equal(reg_test_indices, expected_test_indices)
assert not np.intersect1d(reg_train_indices, reg_test_indices).size
assert len(reg_train_indices) + len(reg_test_indices) == len(ale_df)

X_reg_train_raw = X_reg_all[reg_train_indices]
X_reg_test_raw = X_reg_all[reg_test_indices]
y_reg_train = y_reg_all[reg_train_indices]
y_reg_test = y_reg_all[reg_test_indices]
print("Reused Q4 split: ALE train/test =", len(reg_train_indices), len(reg_test_indices))
print("Classification train/test =", len(class_train_indices), len(class_test_indices))
print("ALE CSV SHA-256:", actual_sha256)

# Exactly one scaler per dataset, fitted only on that dataset's training X.
class_scaler = StandardScaler().fit(X_class_train_raw)
reg_scaler = StandardScaler().fit(X_reg_train_raw)
X_class_train = class_scaler.transform(X_class_train_raw)
X_class_test = class_scaler.transform(X_class_test_raw)
X_reg_train = reg_scaler.transform(X_reg_train_raw)
X_reg_test = reg_scaler.transform(X_reg_test_raw)
assert class_scaler.n_samples_seen_ == len(class_train_indices)
assert reg_scaler.n_samples_seen_ == len(reg_train_indices)
for name, X, y in [
    ("Classification train", X_class_train, y_class_train),
    ("Classification test", X_class_test, y_class_test),
    ("Regression train", X_reg_train, y_reg_train),
    ("Regression test", X_reg_test, y_reg_test),
]:
    assert X.ndim == 2 and y.ndim == 1 and len(X) == len(y)
    assert np.isfinite(X).all() and np.isfinite(y).all()
    print(name, "X/y shapes:", X.shape, y.shape)
print("Regression y retained in the original ale units; no target scaling.")

np.savez(RESULTS_DIR / "classification_train_test_split.npz",
         train_indices=class_train_indices, test_indices=class_test_indices,
         test_size=SPLIT_CONFIG["test_size"], random_state=SPLIT_CONFIG["random_state"],
         stratified=True)

Classification target_names: ['malignant' 'benign']
Original label meanings: {0: np.str_('malignant'), 1: np.str_('benign')}
Classification X shape: (569, 30) y shape: (569,)
Classification feature names: [np.str_('mean radius'), np.str_('mean texture'), np.str_('mean perimeter'), np.str_('mean area'), np.str_('mean smoothness'), np.str_('mean compactness'), np.str_('mean concavity'), np.str_('mean concave points'), np.str_('mean symmetry'), np.str_('mean fractal dimension'), np.str_('radius error'), np.str_('texture error'), np.str_('perimeter error'), np.str_('area error'), np.str_('smoothness error'), np.str_('compactness error'), np.str_('concavity error'), np.str_('concave points error'), np.str_('symmetry error'), np.str_('fractal dimension error'), np.str_('worst radius'), np.str_('worst texture'), np.str_('worst perimeter'), np.str_('worst area'), np.str_('worst smoothness'), np.str_('worst compactness'), np.str_('worst concavity'), np.str_('worst concave points'), np.str_('wor

## Baselines, probability mappings, and training objectives

C0 is `LogisticRegression(max_iter=5000)` with its actual default parameters printed below. R0 is `LinearRegression()`. Both fit the same standardized training inputs as their PyTorch comparisons.

Each self-implemented model has its own zero-initialized linear parameters, $z=Xw+b$. C1 maps $z$ through sigmoid; C2 uses the **standard normal CDF** $\Phi(z)$, with `Normal(loc=0, scale=1)`. Both minimize the same explicit mean binary cross-entropy after clipping probabilities to $[10^{-12},1-10^{-12}]$ for numerical stability. No regularization or class weighting is added.

R1 has a direct linear output and minimizes mean squared error. R2 also has a direct linear output and minimizes `-StudentT(df=5, loc=0, scale=1).log_prob(y-y_pred).mean()`. Its degrees of freedom and scale remain fixed, and only $w,b$ are optimized. MAE is a common **test evaluation metric**, not the training objective of either model. The MSE and Student-t NLL loss magnitudes are not directly comparable.

Implementation references: [PyTorch 2.10 probability distributions (Normal and StudentT)](https://docs.pytorch.org/docs/2.10/distributions.html) and [torch.optim.SGD](https://docs.pytorch.org/docs/2.10/generated/torch.optim.SGD.html).

In [3]:
# Fit both baselines; do not evaluate test metrics until every model has trained.
c0 = LogisticRegression(max_iter=5000)
r0 = LinearRegression()
print("C0 actual parameters:", c0.get_params())
print("R0 actual parameters:", r0.get_params())
c0.fit(X_class_train, y_class_train)
r0.fit(X_reg_train, y_reg_train)
print("Both sklearn baselines fitted. C0 solver iterations:", c0.n_iter_)
(RESULTS_DIR / "sklearn_baseline_parameters.json").write_text(
    json.dumps({"C0": c0.get_params(), "R0": r0.get_params()}, indent=2)
)

DEVICE = torch.device(CONFIG["device"])
DTYPE = CONFIG["dtype"]
Xc_train_t = torch.as_tensor(X_class_train, dtype=DTYPE, device=DEVICE)
yc_train_t = torch.as_tensor(y_class_train, dtype=DTYPE, device=DEVICE)
Xr_train_t = torch.as_tensor(X_reg_train, dtype=DTYPE, device=DEVICE)
yr_train_t = torch.as_tensor(y_reg_train, dtype=DTYPE, device=DEVICE)

standard_normal = torch.distributions.Normal(
    loc=torch.tensor(0.0, dtype=DTYPE, device=DEVICE),
    scale=torch.tensor(1.0, dtype=DTYPE, device=DEVICE),
)
student_t = torch.distributions.StudentT(
    df=torch.tensor(5.0, dtype=DTYPE, device=DEVICE),
    loc=torch.tensor(0.0, dtype=DTYPE, device=DEVICE),
    scale=torch.tensor(1.0, dtype=DTYPE, device=DEVICE),
)

def binary_cross_entropy_explicit(probability, y):
    if probability.shape != y.shape or probability.ndim != 1:
        raise ValueError("BCE probabilities and labels must be matched one-dimensional tensors.")
    eps = CONFIG["probability_clip_eps"]
    p_safe = torch.clamp(probability, min=eps, max=1.0 - eps)
    return -(y * torch.log(p_safe) + (1.0 - y) * torch.log(1.0 - p_safe)).mean()

def logistic_probability(z):
    return torch.sigmoid(z)

def probit_probability(z):
    return standard_normal.cdf(z)

def logistic_objective(z, y):
    return binary_cross_entropy_explicit(logistic_probability(z), y)

def probit_objective(z, y):
    return binary_cross_entropy_explicit(probit_probability(z), y)

def mse_objective(y_pred, y):
    if y_pred.shape != y.shape or y_pred.ndim != 1:
        raise ValueError("Regression predictions and targets must have matching 1D shapes.")
    return ((y - y_pred) ** 2).mean()

def student_t_objective(y_pred, y):
    if y_pred.shape != y.shape or y_pred.ndim != 1:
        raise ValueError("Regression predictions and targets must have matching 1D shapes.")
    residual = y - y_pred
    return -student_t.log_prob(residual).mean()

def require_finite(value, context):
    if not torch.isfinite(value).all().item():
        raise FloatingPointError(f"Non-finite values detected in {context}.")

def initialize_linear(input_dim):
    # Each call creates independent w and b, both initialized to zero.
    model = nn.Linear(input_dim, 1, bias=True, device=DEVICE, dtype=DTYPE)
    with torch.no_grad():
        model.weight.zero_()
        model.bias.zero_()
    return model

C0 actual parameters: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 5000, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
R0 actual parameters: {'copy_X': True, 'fit_intercept': True, 'n_jobs': None, 'positive': False, 'tol': 1e-06}
Both sklearn baselines fitted. C0 solver iterations: [19]


## Automatic differentiation and mini-batch SGD

Every epoch reshuffles the training data using a per-model PyTorch generator seeded with 42. Models within the same dataset therefore receive the same sequence of batch indices. The loop uses `zero_grad`, a tensor-valued mean batch loss, `backward`, and `step`. No NumPy conversion or detachment occurs inside the loss computation.

Each batch checks predictions, loss, gradients, and updated parameters for finite values. At each epoch's end, the current model's average loss is evaluated on the **entire training set**, and recorded. CPU computation uses one thread to reduce overhead for these small models. The scheduled epoch count is fixed; test metrics never determine learning rates, stopping, or other choices. We do not claim that the finite training schedule is an exact optimizer of every objective.

In [4]:
def train_with_sgd(model_name, input_dim, X_train, y_train, objective):
    if X_train.dtype != DTYPE or y_train.dtype != DTYPE:
        raise TypeError("Inputs and labels must use the configured dtype.")
    if X_train.device != DEVICE or y_train.device != DEVICE:
        raise ValueError("Inputs and labels must use the configured device.")
    if y_train.ndim != 1 or len(X_train) != len(y_train):
        raise ValueError("Training labels must have shape (n,).")
    require_finite(X_train, model_name + " training inputs")
    require_finite(y_train, model_name + " training targets")
    model = initialize_linear(input_dim)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        momentum=CONFIG["momentum"],
        weight_decay=CONFIG["weight_decay"],
    )
    generator = torch.Generator(device="cpu")
    generator.manual_seed(CONFIG["seed"])
    with torch.no_grad():
        initial_predictions = model(X_train).squeeze(-1)
        initial_loss_tensor = objective(initial_predictions, y_train)
        require_finite(initial_loss_tensor, model_name + " initial full training loss")
        initial_loss = initial_loss_tensor.item()
    losses = []
    clipping_counts = []
    n_train = len(y_train)
    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        order = torch.randperm(n_train, generator=generator)
        for start in range(0, n_train, CONFIG["batch_size"]):
            indices = order[start:start + CONFIG["batch_size"]].to(DEVICE)
            X_batch, y_batch = X_train[indices], y_train[indices]
            optimizer.zero_grad()
            predictions = model(X_batch).squeeze(-1)
            if predictions.shape != y_batch.shape:
                raise ValueError("Prediction/label shape mismatch would cause incorrect broadcasting.")
            require_finite(predictions, f"{model_name}, epoch {epoch}, batch predictions")
            loss = objective(predictions, y_batch)
            require_finite(loss, f"{model_name}, epoch {epoch}, batch loss")
            loss.backward()
            for parameter_name, parameter in model.named_parameters():
                if parameter.grad is None:
                    raise RuntimeError(f"Missing gradient for {model_name}.{parameter_name}.")
                require_finite(parameter.grad, f"{model_name}, epoch {epoch}, {parameter_name} gradient")
            optimizer.step()
            for parameter_name, parameter in model.named_parameters():
                require_finite(parameter, f"{model_name}, epoch {epoch}, updated {parameter_name}")
        # This diagnostic runs after all batches, using the current model on full train data.
        model.eval()
        with torch.no_grad():
            full_predictions = model(X_train).squeeze(-1)
            require_finite(full_predictions, f"{model_name}, epoch {epoch}, full train predictions")
            full_loss_tensor = objective(full_predictions, y_train)
            require_finite(full_loss_tensor, f"{model_name}, epoch {epoch}, full train loss")
            losses.append(full_loss_tensor.item())
            if model_name.startswith("C"):
                probability = (logistic_probability(full_predictions) if model_name == "C1"
                               else probit_probability(full_predictions))
                require_finite(probability, f"{model_name}, epoch {epoch}, probabilities")
                eps = CONFIG["probability_clip_eps"]
                clipping_counts.append(int(((probability < eps) | (probability > 1.0 - eps)).sum().item()))
        if epoch in (1, 250, 500, 750, CONFIG["epochs"]):
            print(f"{model_name}: epoch {epoch:4d}, full training loss = {losses[-1]:.10f}")
    # Full-training gradient norm is a training-only diagnostic, not a stopping rule.
    optimizer.zero_grad()
    final_train_objective = objective(model(X_train).squeeze(-1), y_train)
    final_train_objective.backward()
    gradient_norm = torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters())).item()
    require_finite(torch.tensor(gradient_norm, dtype=DTYPE), model_name + " final full-training gradient norm")
    optimizer.zero_grad()
    model.eval()
    diagnostics = {
        "model": model_name,
        "epochs": CONFIG["epochs"],
        "initial_full_train_loss": initial_loss,
        "final_full_train_loss": losses[-1],
        "mean_loss_epochs_801_900": float(np.mean(losses[800:900])),
        "mean_loss_epochs_901_1000": float(np.mean(losses[900:1000])),
        "final_full_train_gradient_norm": gradient_norm,
        "finite_checks": "passed for all batches and epochs",
        "max_train_probabilities_clipped_in_an_epoch": max(clipping_counts) if clipping_counts else None,
    }
    return model, np.asarray(losses), diagnostics

# Four independent zero-initialized models: no pretrained coefficients are reused.
c1, c1_losses, c1_diagnostics = train_with_sgd(
    "C1", Xc_train_t.shape[1], Xc_train_t, yc_train_t, logistic_objective
)
c2, c2_losses, c2_diagnostics = train_with_sgd(
    "C2", Xc_train_t.shape[1], Xc_train_t, yc_train_t, probit_objective
)
r1, r1_losses, r1_diagnostics = train_with_sgd(
    "R1", Xr_train_t.shape[1], Xr_train_t, yr_train_t, mse_objective
)
r2, r2_losses, r2_diagnostics = train_with_sgd(
    "R2", Xr_train_t.shape[1], Xr_train_t, yr_train_t, student_t_objective
)

all_models_fitted = True
training_diagnostics = pd.DataFrame([
    c1_diagnostics, c2_diagnostics, r1_diagnostics, r2_diagnostics
])
loss_history = pd.DataFrame({
    "epoch": np.arange(1, CONFIG["epochs"] + 1),
    "C1_logistic_BCE": c1_losses,
    "C2_probit_BCE": c2_losses,
    "R1_MSE": r1_losses,
    "R2_Student_t_NLL": r2_losses,
})
loss_history.to_csv(RESULTS_DIR / "training_loss_by_epoch.csv", index=False, float_format="%.12g")
training_diagnostics.to_csv(RESULTS_DIR / "training_diagnostics.csv", index=False, float_format="%.12g")
print("\nTraining-only diagnostics; different objective magnitudes do not rank model quality:")
display(training_diagnostics)
print("All six models have been fitted. No test metric has been evaluated yet.")

C1: epoch    1, full training loss = 0.4901660171


C1: epoch  250, full training loss = 0.0686479050


C1: epoch  500, full training loss = 0.0595789637


C1: epoch  750, full training loss = 0.0555145560


C1: epoch 1000, full training loss = 0.0530684950
C2: epoch    1, full training loss = 0.3507047893


...

C2: epoch 1000, full training loss = 0.0470547440
R1: epoch    1, full training loss = 0.9517294155
R1: epoch  250, full training loss = 0.0505849859
R1: epoch  500, full training loss = 0.0505925624
R1: epoch  750, full training loss = 0.0505861476


R1: epoch 1000, full training loss = 0.0505858724
R2: epoch    1, full training loss = 1.4903249607
R2: epoch  250, full training loss = 0.9976683602
R2: epoch  500, full training loss = 0.9976639694


R2: epoch  750, full training loss = 0.9976622872
R2: epoch 1000, full training loss = 0.9976623476

Training-only diagnostics; different objective magnitudes do not rank model quality:


,model,epochs,initial_full_train_loss,final_full_train_loss,mean_loss_epochs_801_900,mean_loss_epochs_901_1000,final_full_train_gradient_norm,finite_checks,max_train_probabilities_clipped_in_an_epoch
0,C1,1000,0.693147,0.053068,0.054423,0.053490,0.007321,passed for all batches and epochs,6.0
1,C2,1000,0.693147,0.047055,0.048027,0.047362,0.006381,passed for all batches and epochs,106.0
2,R1,1000,1.066105,0.050586,0.050588,0.050584,0.004176,passed for all batches and epochs,NaN
3,R2,1000,1.516088,0.997662,0.997663,0.997662,0.001637,passed for all batches and epochs,NaN


All six models have been fitted. No test metric has been evaluated yet.


## Final held-out evaluation

Only now, after all six models have fitted, do we compute test metrics. PyTorch models use `eval()` and `torch.no_grad()`. Classification predictions retain the original labels and use the fixed threshold `p >= 0.5`. Regression predictions and targets are both one-dimensional, in identical sample order; all MAEs remain in the original `ale` units.

In [5]:
assert all_models_fitted
Xc_test_t = torch.as_tensor(X_class_test, dtype=DTYPE, device=DEVICE)
Xr_test_t = torch.as_tensor(X_reg_test, dtype=DTYPE, device=DEVICE)
for model in (c1, c2, r1, r2):
    model.eval()
with torch.no_grad():
    c1_prob_t = logistic_probability(c1(Xc_test_t).squeeze(-1))
    c2_prob_t = probit_probability(c2(Xc_test_t).squeeze(-1))
    r1_pred_t = r1(Xr_test_t).squeeze(-1)
    r2_pred_t = r2(Xr_test_t).squeeze(-1)
    for name, value in [("C1 test probabilities", c1_prob_t), ("C2 test probabilities", c2_prob_t),
                        ("R1 test predictions", r1_pred_t), ("R2 test predictions", r2_pred_t)]:
        require_finite(value, name)

c1_prob = c1_prob_t.cpu().numpy().reshape(-1)
c2_prob = c2_prob_t.cpu().numpy().reshape(-1)
class_predictions = {
    "C0 — sklearn LogisticRegression": np.asarray(c0.predict(X_class_test)).reshape(-1),
    "C1 — Logistic + BCE + SGD": (c1_prob >= 0.5).astype(np.int64),
    "C2 — Probit + BCE + SGD": (c2_prob >= 0.5).astype(np.int64),
}
reg_predictions = {
    "R0 — sklearn LinearRegression": np.asarray(r0.predict(X_reg_test)).reshape(-1),
    "R1 — Linear + MSE + SGD": r1_pred_t.cpu().numpy().reshape(-1),
    "R2 — Linear + Student-t NLL + SGD": r2_pred_t.cpu().numpy().reshape(-1),
}
classification_rows = []
for name, prediction in class_predictions.items():
    assert prediction.ndim == 1 and prediction.shape == y_class_test.shape
    assert np.isfinite(prediction).all()
    classification_rows.append({
        "Model": name,
        "Test Accuracy": accuracy_score(y_class_test, prediction),
        "Correct / Test Samples": f"{int((prediction == y_class_test).sum())} / {len(y_class_test)}",
    })
objectives = {
    "R0 — sklearn LinearRegression": "Ordinary least squares (native sklearn fit)",
    "R1 — Linear + MSE + SGD": "MSE",
    "R2 — Linear + Student-t NLL + SGD": "Student-t NLL (df=5, loc=0, scale=1)",
}
regression_rows = []
for name, prediction in reg_predictions.items():
    assert prediction.ndim == 1 and prediction.shape == y_reg_test.shape
    assert np.isfinite(prediction).all()
    mae = mean_absolute_error(y_reg_test, prediction)
    assert np.isclose(mae, np.mean(np.abs(y_reg_test - prediction)))
    regression_rows.append({"Model": name, "Training Objective": objectives[name], "Test MAE": mae})
classification_results = pd.DataFrame(classification_rows)
regression_results = pd.DataFrame(regression_rows)
print("Classification results (original 0/1 labels; threshold 0.5 for C1/C2):")
display(classification_results.style.format({"Test Accuracy": "{:.8f}"}).hide(axis="index"))
print("Regression results (MAE in original ale units; n_test =", len(y_reg_test), "):")
display(regression_results.style.format({"Test MAE": "{:.10f}"}).hide(axis="index"))
classification_results.to_csv(RESULTS_DIR / "classification_results.csv", index=False, float_format="%.12g")
regression_results.to_csv(RESULTS_DIR / "regression_results.csv", index=False, float_format="%.12g")

# Save predictions with their source row indices, preserving test order.
pd.DataFrame({
    "dataset_row_index": class_test_indices,
    "y_true_original_label": y_class_test,
    "C0_predicted_label": list(class_predictions.values())[0],
    "C1_probability_label_1": c1_prob,
    "C1_predicted_label": list(class_predictions.values())[1],
    "C2_probability_label_1": c2_prob,
    "C2_predicted_label": list(class_predictions.values())[2],
}).to_csv(RESULTS_DIR / "classification_test_predictions.csv", index=False, float_format="%.12g")
pd.DataFrame({
    "dataset_row_index": reg_test_indices,
    "observed_ale": y_reg_test,
    "R0_predicted_ale": list(reg_predictions.values())[0],
    "R1_predicted_ale": list(reg_predictions.values())[1],
    "R2_predicted_ale": list(reg_predictions.values())[2],
}).to_csv(RESULTS_DIR / "regression_test_predictions.csv", index=False, float_format="%.12g")

Classification results (original 0/1 labels; threshold 0.5 for C1/C2):


Model,Test Accuracy,Correct / Test Samples
C0 — sklearn LogisticRegression,0.98245614,112 / 114
C1 — Logistic + BCE + SGD,0.96491228,110 / 114
C2 — Probit + BCE + SGD,0.97368421,111 / 114


Regression results (MAE in original ale units; n_test = 22 ):


Model,Training Objective,Test MAE
R0 — sklearn LinearRegression,Ordinary least squares (native sklearn fit),0.1953934164
R1 — Linear + MSE + SGD,MSE,0.1952354916
R2 — Linear + Student-t NLL + SGD,"Student-t NLL (df=5, loc=0, scale=1)",0.1940189437


## English conclusions for the four subquestions

The following text is generated from the actual measured results above. The comparisons concern the one fixed split and are not used to select training settings.

In [6]:
c0_acc, c1_acc, c2_acc = classification_results["Test Accuracy"].to_numpy()
r0_mae, r1_mae, r2_mae = regression_results["Test MAE"].to_numpy()

classification_a = (
    f"**Classification (a).** The self-implemented sigmoid logistic regression, trained with "
    f"explicit BCE and mini-batch SGD, achieved test accuracy **{c1_acc:.8f}** "
    f"({classification_results.iloc[1]['Correct / Test Samples']} correct), compared with "
    f"**{c0_acc:.8f}** ({classification_results.iloc[0]['Correct / Test Samples']} correct) "
    f"for scikit-learn LogisticRegression. The accuracy difference (C1 minus C0) is "
    f"**{c1_acc - c0_acc:+.8f}**. The baseline retains its default regularization, "
    f"whereas C1 has no regularization term; their objectives and optimization procedures differ, "
    f"so exact agreement is not required. These differences could affect the results, "
    f"but this experiment does not isolate their individual effects."
)
classification_b = (
    f"**Classification (b).** The self-implemented probit model used the standard normal CDF "
    f"and the same explicit BCE and SGD schedule. Its test accuracy was **{c2_acc:.8f}** "
    f"({classification_results.iloc[2]['Correct / Test Samples']} correct), compared with "
    f"the scikit-learn logistic baseline's **{c0_acc:.8f}**. The accuracy difference "
    f"(C2 minus C0) is **{c2_acc - c0_acc:+.8f}**. This is the observed comparison "
    f"on this split; it does not establish that the probit link is generally better or worse."
)
regression_a = (
    f"**Regression (a).** The self-implemented linear model trained with MSE and mini-batch SGD "
    f"obtained test MAE **{r1_mae:.10f}**, compared with **{r0_mae:.10f}** for "
    f"scikit-learn LinearRegression. The MAE difference (R1 minus R0) is "
    f"**{r1_mae - r0_mae:+.10f}**, in the original ale units. Both models fit a linear "
    f"prediction function using a squared-error objective, while R1 uses a finite stochastic "
    f"optimization schedule. The observed difference is compatible with different fitting "
    f"procedures, without proving a specific cause. MSE was the training loss; MAE was used only "
    f"for held-out evaluation."
)
regression_b = (
    f"**Regression (b).** The linear model trained with fixed unit-scale Student-t NLL "
    f"(df=5) achieved test MAE **{r2_mae:.10f}**, compared with the scikit-learn linear "
    f"baseline's **{r0_mae:.10f}** and the MSE-SGD model's **{r1_mae:.10f}**. "
    f"The MAE difference (R2 minus R0) is **{r2_mae - r0_mae:+.10f}**. This comparison "
    f"uses the same test MAE, rather than comparing numerical values of MSE and Student-t NLL. "
    f"Student-t NLL gives large residuals a different influence during fitting, but these results "
    f"alone do not establish the cause of the MAE difference or general superiority."
)
loss_diagnostics_text = (
    f"**Training and scope.** All four self-implemented models started at w=0, b=0 and used "
    f"{CONFIG['epochs']} epochs, learning rate {CONFIG['learning_rate']}, batch size "
    f"{CONFIG['batch_size']}, float64 on CPU, momentum=0, and weight_decay=0. BCE probabilities "
    f"were clipped at {CONFIG['probability_clip_eps']:.0e}; the maximum number of clipped "
    f"training probabilities in one epoch was {c1_diagnostics['max_train_probabilities_clipped_in_an_epoch']} "
    f"for C1 and {c2_diagnostics['max_train_probabilities_clipped_in_an_epoch']} for C2, out of "
    f"{len(y_class_train)} training samples. Both targets and parameters retained "
    f"the configured dtype. All batch/epoch finite-value checks passed. The final full-training "
    f"losses were C1={c1_losses[-1]:.10f}, C2={c2_losses[-1]:.10f}, "
    f"R1={r1_losses[-1]:.10f}, and R2={r2_losses[-1]:.10f}. These are four objective values, "
    f"not four interchangeable model-quality scores. No training configuration was selected "
    f"using test metrics. A finite SGD schedule need not reach an exact optimum; the recorded "
    f"per-epoch losses and training gradient norms document its optimization progress. "
    f"The classifier losses were still slowly decreasing at the end of the schedule, "
    f"while the regression losses were close to a plateau; exact convergence is not claimed. "
    f"The test sets contain {len(y_class_test)} classification samples and only "
    f"{len(y_reg_test)} ALE regression samples. All comparisons are specific to these current "
    f"samples, the fixed split, and this schedule."
)
conclusion_text = "\n\n".join([
    classification_a, classification_b, regression_a, regression_b, loss_diagnostics_text
])
display(Markdown(conclusion_text))
(RESULTS_DIR / "english_conclusions.txt").write_text(conclusion_text)
print("\nSaved numeric tables, predictions, training diagnostics, and conclusions to:", RESULTS_DIR)

**Classification (a).** The self-implemented sigmoid logistic regression, trained with explicit BCE and mini-batch SGD, achieved test accuracy **0.96491228** (110 / 114 correct), compared with **0.98245614** (112 / 114 correct) for scikit-learn LogisticRegression. The accuracy difference (C1 minus C0) is **-0.01754386**. The baseline retains its default regularization, whereas C1 has no regularization term; their objectives and optimization procedures differ, so exact agreement is not required. These differences could affect the results, but this experiment does not isolate their individual effects.

**Classification (b).** The self-implemented probit model used the standard normal CDF and the same explicit BCE and SGD schedule. Its test accuracy was **0.97368421** (111 / 114 correct), compared with the scikit-learn logistic baseline's **0.98245614**. The accuracy difference (C2 minus C0) is **-0.00877193**. This is the observed comparison on this split; it does not establish that the probit link is generally better or worse.

**Regression (a).** The self-implemented linear model trained with MSE and mini-batch SGD obtained test MAE **0.1952354916**, compared with **0.1953934164** for scikit-learn LinearRegression. The MAE difference (R1 minus R0) is **-0.0001579248**, in the original ale units. Both models fit a linear prediction function using a squared-error objective, while R1 uses a finite stochastic optimization schedule. The observed difference is compatible with different fitting procedures, without proving a specific cause. MSE was the training loss; MAE was used only for held-out evaluation.

**Regression (b).** The linear model trained with fixed unit-scale Student-t NLL (df=5) achieved test MAE **0.1940189437**, compared with the scikit-learn linear baseline's **0.1953934164** and the MSE-SGD model's **0.1952354916**. The MAE difference (R2 minus R0) is **-0.0013744727**. This comparison uses the same test MAE, rather than comparing numerical values of MSE and Student-t NLL. Student-t NLL gives large residuals a different influence during fitting, but these results alone do not establish the cause of the MAE difference or general superiority.

**Training and scope.** All four self-implemented models started at w=0, b=0 and used 1000 epochs, learning rate 0.01, batch size 32, float64 on CPU, momentum=0, and weight_decay=0. BCE probabilities were clipped at 1e-12; the maximum number of clipped training probabilities in one epoch was 6 for C1 and 106 for C2, out of 455 training samples. Both targets and parameters retained the configured dtype. All batch/epoch finite-value checks passed. The final full-training losses were C1=0.0530684950, C2=0.0470547440, R1=0.0505858724, and R2=0.9976623476. These are four objective values, not four interchangeable model-quality scores. No training configuration was selected using test metrics. A finite SGD schedule need not reach an exact optimum; the recorded per-epoch losses and training gradient norms document its optimization progress. The classifier losses were still slowly decreasing at the end of the schedule, while the regression losses were close to a plateau; exact convergence is not claimed. The test sets contain 114 classification samples and only 22 ALE regression samples. All comparisons are specific to these current samples, the fixed split, and this schedule.


Saved numeric tables, predictions, training diagnostics, and conclusions to: /Users/chongenhu/Documents/New project/ml1_hw1_q5/results
